In [ ]:
import requests

url = "https://openapi.m-pesa.com/sandbox/ipg/v2/vodacomLES/c2bPayment/singleStage/"

session_id = encrypted_session_key

body = {
    "input_Amount": "10",
    "input_Country": "LES",
    "input_Currency": "LSL",
    "input_CustomerMSISDN": "26656237645",
    "input_ServiceProviderCode": "ORG001",
    "input_ThirdPartyConversationID": "123456789", #GENERATE
    "input_TransactionReference": "REF123456",  #GENERATE
    "input_PurchasedItemsDesc": "Test Payment"
}

headers = {
    "Authorization": f"Bearer {session_id}",
    "Content-Type": "application/json",
    "Origin": "http://127.0.0.1"
}

response = requests.post(
    url,
    json=body,
    headers=headers
)

print("Status Code:", response.status_code)
print("Response:", response.text)

<div style="background: linear-gradient(90deg,#0f172a,#1e293b,#334155);
            padding:25px;
            border-radius:10px;
            color:white;">
            
<h1>🚀 Customer Transactions ETL Pipeline</h1>

<h3>Data Engineering Project</h3>

<p>
Source Systems → Data Ingestion/Extraction → Transformation → Validation → Data Warehouse
</p>

</div>

### Module imports

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, explode, trim,lower
from pyspark.sql.functions import regexp_replace, col
from pyspark.sql.functions import array_contains
from pyspark.sql.functions import year, month, dayofmonth, to_timestamp
from pyspark.sql.functions import col, split, explode, trim, broadcast, lower
#from pyspark.sql.functions import to_timestamp
import os
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

In [2]:
spark = SparkSession.builder \
    .appName("Transactions_etl") \
    .getOrCreate()

<div style="background: linear-gradient(90deg,#0f172a,#1e293b,#334155);
            padding:25px;
            border-radius:10px;
            color:white;">
            


<h3>Extraction</h3>


</div>

In [3]:
raw_data_set = spark.read.csv(
    r"C:\Users\patsi\Documents\Virtual_data_department\Data_set\Retail_Transactions_Dataset.csv",
    header=True,
    inferSchema=True
)

pricing_matrix = spark.read.csv(
    r"C:\Users\patsi\Documents\Virtual_data_department\Data_engineering\Extract\products_pricing_matrix.csv",
    header=True,
    inferSchema=True
)

## Transform

In [ ]:
pricing_matrix = pricing_matrix.withColumn(
    "Product_Clean",
    lower(trim(col("Product_Clean")))
)

In [7]:
raw_data_set = raw_data_set.withColumn(
    "Date",
    to_timestamp(col("Date"), "yyyy-MM-dd HH:mm:ss")
)
raw_data_set = (
    raw_data_set
    .withColumn("Day", dayofmonth(col("Date")))
    .withColumn("Month", month(col("Date")))
    .withColumn("Year", year(col("Date")))
)

In [8]:
# 3. Split products into array
raw_data_set_split = raw_data_set.withColumn(
    "Product",
    split("Product", ",")
)

# 4. Explode into rows
raw_data_set_exploded = raw_data_set_split.withColumn(
    "Product",
    explode("Product")
)

# 5. Clean whitespace (best practice)
raw_data_set_final = raw_data_set_exploded.withColumn("Product", trim("Product")) \
                      .select("Transaction_ID", "Product")

In [9]:
def keep_alphanumerics_and_spaces(df, column_name, new_column_name=None):
    if new_column_name is None:
        new_column_name = column_name
    return df.withColumn(
        new_column_name,
        regexp_replace(col(column_name), r"[^a-zA-Z0-9 ]", "")
    )

In [10]:
raw_data_set_product_clean = keep_alphanumerics_and_spaces(raw_data_set_final, "Product", "Product_Clean")

In [ ]:
print("Transactions:", raw_data_set.count())
print("Products:", raw_data_set_product_clean.count())

In [13]:
# Step 1: Normalize + explode transactions (turn list into rows)
transactions_exploded = (
    raw_data_set
    .withColumn(
        "Matched_Product",
        explode(split(col("Product"), ","))
    )
    .withColumn(
        "Matched_Product",
        lower(trim(col("Matched_Product")))
    )
)

In [14]:
transactions_exploded = keep_alphanumerics_and_spaces(transactions_exploded, "Matched_Product", "Matched_Product")

In [15]:
# Step 2: Clean product reference table
products_clean = (
    raw_data_set_product_clean
    .withColumn("Product_Clean", lower(trim(col("Product_Clean"))))
)
transactions_exploded = (
    transactions_exploded
    .withColumn("Matched_Product", lower(trim(col("Matched_Product"))))
)

In [16]:
transactions_exploded_priced = transactions_exploded.join(
    pricing_matrix,
    transactions_exploded.Matched_Product == pricing_matrix.Product_Clean,
    "inner"
)

In [ ]:
transactions_exploded_priced.write \
    .format("jdbc") \
    .option("url", "jdbc:mysql://<HOST>:3306/<DB_NAME>") \
    .option("dbtable", "your_table_name") \
    .option("user", "<USERNAME>") \
    .option("password", "<PASSWORD>") \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .mode("overwrite") \
    .save()